# Portfolio Data Collections

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [141]:
def portfolio_analytics(tickers, risk_free_rate, n_trading_days, port_type = None):
    import yfinance as yf
    import pandas as pd
    import numpy as np
    rng = np.random.default_rng(1000)
    
    # Loading the data 
    df_close = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Close']
    df_vol = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Volume']
    df_mcap = df_close * df_vol


    # Wrangling the data
    drop_10na_cols = df_close.columns[df_close.isna().sum() > 10]
    df_close = df_close.drop(columns=drop_10na_cols)

    # Log returns
    df_returns = np.log(df_close / df_close.shift(1)).dropna()
    n_assets = df_returns.shape[1]
    
    # Weighting techniques
    if port_type == 'EW':
        weights = np.ones(n_assets) / n_assets
    elif port_type == 'MC':
        weights = df_mcap.iloc[-1:,].values / df_mcap.iloc[-1:,].sum(axis = 1).values
    elif port_type == 'RP':
        weights = df_returns.std().values
        weights /= weights.sum()
    else:
        weights = rng.normal(size=n_assets)
        weights /= weights.sum()

    # Portfolio Returns
    cov_mat = df_returns.cov()
    port_returns =  df_returns @ weights
    port_variance = weights.T @ cov_mat @ weights

    # Annualized Returns
    annual_returns = df_returns.mean() * n_trading_days
    annual_std = np.sqrt(port_variance * n_trading_days)
    annual_sharpe = (annual_returns - risk_free_rate) / annual_std

    annual_results = pd.DataFrame({
        'Annual Port Return': [annual_returns],
        'Annual Port Std': [annual_std],
        'Annual Port Sharpe': [annual_sharpe]
    })

    # Risk Attribution
    overall_port_std = np.sqrt(weights.T @ cov_mat @ weights)
    marginal_risk_contribution = (cov_mat @ weights) / overall_port_std
    component_risk_contribution = weights * marginal_risk_contribution
    pct_risk_contribution = component_risk_contribution / component_risk_contribution.sum()
    
    risk_attribution = pd.DataFrame({
        'Asset': n_assets,
        'Weight': weights,
        'Marginal Risk Contribution': marginal_risk_contribution,
        'Component Risk Contribution': component_risk_contribution,
        '% of Total Risk': pct_risk_contribution
    })


    return risk_attribution, annual_results

risk_free_rate = 0.02
n_trading_days = 252
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'TSLA']
port_type = ['EW', 'MC', 'RP', 'OO']



risk_results , annual_results = portfolio_analytics(tickers=tickers, risk_free_rate=risk_free_rate, n_trading_days=n_trading_days, port_type='RP')

risk_results.sort_values('% of Total Risk', ascending=False)

[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  5 of 5 completed


,Asset,Weight,Marginal Risk Contribution,Component Risk Contribution,% of Total Risk
Ticker,,,,,
TSLA,5,0.312658,0.035208,0.011008,0.489896
META,5,0.211745,0.019903,0.004214,0.187551
AMZN,5,0.170215,0.016580,0.002822,0.125592
GOOG,5,0.154780,0.014558,0.002253,0.100276
AAPL,5,0.150602,0.014426,0.002173,0.096685


In [ ]:
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'TSLA']

df_close = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Close']
df_close.columns = tickers
df_returns = np.log(df_close / df_close.shift(1)).dropna()
df_vol = yf.download(tickers=tickers, start='2020-01-01', auto_adjust=True)['Volume']
df_vol.columns = tickers
df_mcap = df_close * df_vol
df_mcap.columns = tickers


[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  5 of 5 completed


In [95]:
drop_10na_cols = df_close.columns[df_close.isna().sum() > 10]
df_close = df_close.drop(columns=drop_10na_cols)
df_close

,AAPL,AMZN,GOOG,META,TSLA
Date,,,,,
2020-01-02,72.333862,94.900497,67.811768,208.146561,28.684000
2020-01-03,71.630630,93.748497,67.479004,207.045227,29.534000
2020-01-06,72.201439,95.143997,69.142838,210.944626,30.102667
2020-01-07,71.861824,95.343002,69.099693,211.401001,31.270666
2020-01-08,73.017838,94.598503,69.644226,213.544220,32.809334
...,...,...,...,...,...
2026-06-02,315.200012,256.519989,358.390015,597.630005,423.739990
2026-06-03,310.260010,250.020004,355.679993,622.979980,423.700012
2026-06-04,311.230011,253.789993,369.269989,627.570007,418.450012


In [ ]:
df_returns

,AAPL,AMZN,GOOG,META,TSLA
Date,,,,,
2020-01-03,-0.009770,-0.012213,-0.004919,-0.005305,0.029203
2020-01-06,0.007937,0.014776,0.024358,0.018658,0.019072
2020-01-07,-0.004715,0.002089,-0.000624,0.002161,0.038067
2020-01-08,0.015959,-0.007839,0.007850,0.010087,0.048033
2020-01-09,0.021019,0.004788,0.010984,0.014210,-0.022189
...,...,...,...,...,...
2026-06-02,0.028610,-0.018310,-0.038830,-0.004741,0.018723
2026-06-03,-0.015797,-0.025666,-0.007590,0.041543,-0.000094
2026-06-04,0.003122,0.014966,0.037497,0.007341,-0.012468


In [139]:
weights = df_returns.std().values
weights /= weights.sum()
n_trading_days = 252
risk_free_rate = 0.02

asset_returns_annual = df_returns.mean() * n_trading_days
asset_std_annual = df_returns.std() * np.sqrt(n_trading_days)
asset_sharpe_annual = asset_returns_annual - risk_free_rate / asset_std_annual

port_returns_annual = (df_returns @ weights).mean()*n_trading_days
component_return_contribution = weights * asset_returns_annual
pct_return_contribution = component_return_contribution / port_returns_annual


print(f'Annual Asset Sharpe: {asset_sharpe_annual}')
print(f'% of Portfolio Returns: {pct_return_contribution:.2f}')


Annual Asset Sharpe: AAPL    0.164122
AMZN    0.093141
GOOG    0.199489
META    0.115750
TSLA    0.379950
dtype: float64


TypeError: unsupported format string passed to Series.__format__

In [130]:
weights * asset_returns_annual

AAPL    0.034345
AMZN    0.025485
GOOG    0.040503
META    0.034141
TSLA    0.128427
dtype: float64

In [ ]:
df_mcap.iloc[-1:,].values / df_mcap.iloc[-1:,].sum(axis = 1).values

array([[0.26561128, 0.13016969, 0.12379125, 0.21593573, 0.26449206]])

In [ ]:
df_mcap

,AAPL,AMZN,GOOG,META,TSLA
Date,,,,,
2020-01-02,9.799821e+09,7.647082e+09,1.907681e+09,2.513807e+09,4.101281e+09
2020-01-03,1.048119e+10,7.058137e+09,1.601142e+09,2.316505e+09,7.876053e+09
2020-01-06,8.547726e+09,7.729118e+09,2.395523e+09,3.598483e+09,4.575455e+09
2020-01-07,7.823741e+09,7.713058e+09,2.076722e+09,3.152496e+09,8.387778e+09
2020-01-08,9.644138e+09,6.637031e+09,2.128328e+09,2.877508e+09,1.532736e+10
...,...,...,...,...,...
2026-06-02,1.403734e+10,1.072536e+10,1.241771e+10,1.090794e+10,1.593093e+10
2026-06-03,1.577260e+10,1.284958e+10,1.530527e+10,1.434106e+10,1.885495e+10
2026-06-04,1.396461e+10,9.037411e+09,1.393854e+10,1.347343e+10,1.478104e+10


In [ ]:
df_mcap.iloc[-1:,].values / df_mcap.iloc[-1:,].sum(axis = 1).values

array([[0.26561128, 0.13016969, 0.12379125, 0.21593573, 0.26449206]])

In [75]:
cov_mat = df_returns.cov()
weights = df_returns.std().values
weights /= weights.sum()
weights

array([0.15058756, 0.17022521, 0.15476072, 0.21176128, 0.31266523])

In [79]:
weights @ cov_mat

AAPL    0.000324
AMZN    0.000373
GOOG    0.000327
META    0.000447
TSLA    0.000791
dtype: float64